The same seq2seq skills cover other tasks too, such as summarization, paraphrasing, and code translation (Python → JS). Once you’ve learned translation, those come almost for free.

In [87]:
%pip install sentencepiece sacremoses

Note: you may need to restart the kernel to use updated packages.


In [88]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-fr", device_map="auto")

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 985.22it/s]


In [89]:
from datasets import load_dataset

raw_datasets = load_dataset("Helsinki-NLP/kde4", lang1='en', lang2='fr', trust_remote_code=True)

In [90]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 210173
    })
})

In [91]:
split_datasets = raw_datasets['train'].train_test_split(train_size=0.9, seed=42)
split_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 189155
    })
    test: Dataset({
        features: ['id', 'translation'],
        num_rows: 21018
    })
})

In [92]:
split_datasets['validation'] = split_datasets.pop('test')

In [93]:
split_datasets, split_datasets['train'][0]

(DatasetDict({
     train: Dataset({
         features: ['id', 'translation'],
         num_rows: 189155
     })
     validation: Dataset({
         features: ['id', 'translation'],
         num_rows: 21018
     })
 }),
 {'id': '118328',
  'translation': {'en': 'Show Pathfinder Lander Image',
   'fr': "Afficher l'image de Pathfinder LanderImage/ info menu item (should be translated)"}})

In [94]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [95]:
inputs = tokenizer('email', return_tensors='pt').to(model.device)

In [96]:
model

MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(59514, 512, padding_idx=59513)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(59514, 512, padding_idx=59513)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,)

In [97]:
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=64, num_beams=4)
print(tokenizer.batch_decode(output_ids, skip_special_tokens=True))
# ['e-mail'] (or similar)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['Courriel']


In [98]:
en_sentence = split_datasets["train"][1]["translation"]["en"]
fr_sentence = split_datasets["train"][1]["translation"]["fr"]

inputs = tokenizer(en_sentence, text_target=fr_sentence)
inputs

{'input_ids': [34014, 0], 'attention_mask': [1, 1], 'labels': [39433, 1547, 0]}

In [99]:
wrong_targets = tokenizer(fr_sentence)
print(tokenizer.convert_ids_to_tokens(wrong_targets["input_ids"]))
print(tokenizer.convert_ids_to_tokens(inputs["labels"]))

['▁É', 'd', 'ite', 'ur', '</s>']
['▁Édit', 'eur', '</s>']


In [100]:
max_length = 128

def preprocess(example):
    inputs = [ex['en'] for ex in example['translation']]
    target = [ex['fr'] for ex in example['translation']]
    model_inputs = tokenizer(inputs, text_target=target, max_length=max_length, truncation=True)

    return model_inputs

In [101]:
tokenized_datasets = split_datasets.map(
    preprocess, batched=True, remove_columns=split_datasets["train"].column_names
)

In [102]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [103]:
batch = data_collator([tokenized_datasets["train"][i] for i in range(1, 3)])
batch.keys()

KeysView({'input_ids': tensor([[34014,     0, 59513],
        [13331,  3280,     0]]), 'attention_mask': tensor([[1, 1, 0],
        [1, 1, 1]]), 'labels': tensor([[39433,  1547,     0,  -100,  -100],
        [24251,    14,     6,  4930,     0]]), 'decoder_input_ids': tensor([[59513, 39433,  1547,     0, 59513],
        [59513, 24251,    14,     6,  4930]])})

In [104]:
for i in range(1, 3):
    print(tokenized_datasets["train"][i]["labels"])

[39433, 1547, 0]
[24251, 14, 6, 4930, 0]


In [105]:
%pip install sacrebleu

Note: you may need to restart the kernel to use updated packages.


In [106]:
import evaluate

metric = evaluate.load('sacrebleu')

In [107]:
predictions = [
    "This plugin lets you translate web pages between several languages automatically."
]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

{'score': 46.750469682990186,
 'counts': [11, 6, 4, 3],
 'totals': [12, 11, 10, 9],
 'precisions': [91.66666666666667,
  54.54545454545455,
  40.0,
  33.333333333333336],
 'bp': 0.9200444146293233,
 'sys_len': 12,
 'ref_len': 13}

In [108]:
predictions = ["This This This This"]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

{'score': 1.683602693167689,
 'counts': [1, 0, 0, 0],
 'totals': [4, 3, 2, 1],
 'precisions': [25.0, 16.666666666666668, 12.5, 12.5],
 'bp': 0.10539922456186433,
 'sys_len': 4,
 'ref_len': 13}

In [109]:
import numpy as np

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [d.strip() for d in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]

    results = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {
        'bleu': results['score']
    }

In [110]:
from huggingface_hub import notebook_login

notebook_login()

In [111]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    f"marian-finetuned-kde4-en-to-fr",
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=True,
)

In [112]:
from transformers import Seq2SeqTrainer

# sort validation by input length so each generation batch has similar-length sentences (less padding, same results)
eval_dataset = (
    tokenized_datasets["validation"]
    .map(lambda x: {"len": len(x["input_ids"])})
    .sort("len", reverse=True)
    .remove_columns("len")
)

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Map: 100%|██████████| 21018/21018 [00:00<00:00, 44534.06 examples/s]
C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\pyarrow\compute.py:230: FutureWarning: Specifying null_placement in SortOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)


In [113]:
import torch
print(torch.cuda.is_available(), trainer.model.device)

True cuda:0


In [114]:
trainer.evaluate(max_length=max_length)

Training Loss,Validation Loss,Epoch,Bleu
No log,2.280360,0,39.393114


{'eval_loss': 2.280359983444214, 'eval_bleu': 39.39311370420435}

In [115]:
trainer.train()

Epoch,Training Loss,Validation Loss,Bleu
1,1.019577,1.050693,51.346260
2,0.869265,0.978927,52.806030
3,0.827064,0.960619,53.349281


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]


TrainOutput(global_step=17736, training_loss=0.9365133510068241, metrics={'train_runtime': 3958.977, 'train_samples_per_second': 143.336, 'train_steps_per_second': 4.48, 'total_flos': 1.1304295115194368e+16, 'train_loss': 0.9365133510068241, 'epoch': 3.0})